# YLM: official TensorQA-v0 multi-seed Options A/B/C and sample-efficiency runner

Select a GPU runtime. By default (`YLM_FULL=1`, `YLM_DATA_SOURCE=v0_release`), this notebook downloads and verifies the official `v0.1.0` `tensorqa_v0.tar.gz` release (`84,757` questions, `584` unique strings; pinned SHA-256 digests in `DATA_CARD.md`) instead of rebuilding with `tblite`, so results can sit directly next to the official v0 benchmark table.

Two full GPU workflows are supported via `YLM_MODE`:
- `options` (default, Issue #56): trains Options A, B, B (`--attn_sup 0.1`), and C across multiple seeds (`YLM_SEEDS='0,1,2'`) with `--eval_rotations 32` (no reflections) and writes a provenance-verified table (`results_options_v0.md`).
- `fractions` (Issue #58): trains the `3% / 10% / 30%` data-fraction curves (`minilm`, `matscibert`, `taskid`) for extra seeds (`YLM_SEEDS='1,2'`) at `--epochs 150 --eval_rotations 32` and combines them with `runs_slurm` seed 0 into `results.md` and `sample_efficiency.png`.

All data and checkpoints persist in Google Drive (`MyDrive/YLM_runs/v0_multiseed`). Rerunning after a disconnect skips finished runs (`results.json` present) and resumes interrupted runs from `latest.pt`.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys, time

# ---- Settings (plain assignments; override via os.environ before running) ----
def _flag(name, default):
    return os.environ.get(name, str(default)).strip().lower() in ('1', 'true', 'yes')

FULL = _flag('YLM_FULL', True)
MODE = os.environ.get('YLM_MODE', 'options').strip().lower()  # 'options' (#56) or 'fractions' (#58)
DATA_SOURCE = os.environ.get('YLM_DATA_SOURCE', 'v0_release').strip().lower()  # 'v0_release' or 'rebuild'
BRANCH = os.environ.get('YLM_BRANCH', 'main')

# Options mode (#56) settings
OPTIONS_TO_RUN = os.environ.get('YLM_OPTIONS', 'A,B,C')
TEXT_ENCODER = os.environ.get('YLM_TEXT', 'minilm')
INCLUDE_B_ATTN_SUP = _flag('YLM_INCLUDE_B_ATTN_SUP', True)
ATTN_SUP = float(os.environ.get('YLM_ATTN_SUP', '0.1'))
C_TP_MODE = os.environ.get('YLM_C_TP_MODE', 'fully_connected')  # or 'channelwise'

# Fractions mode (#58) settings
FRACTIONS = os.environ.get('YLM_FRACTIONS', '0.03,0.1,0.3')
FRACTION_TEXTS = os.environ.get('YLM_FRACTION_TEXTS', 'minilm,matscibert,taskid')

# Shared evaluation & seed protocol (matches official v0: 32 SO(3) rotations)
DEFAULT_SEEDS = '0,1,2' if MODE == 'options' else '1,2'
SEEDS = [int(s.strip()) for s in os.environ.get('YLM_SEEDS', DEFAULT_SEEDS).split(',') if s.strip()]
EVAL_ROTATIONS = int(os.environ.get('YLM_EVAL_ROTATIONS', '32' if FULL else '4'))
EVAL_REFLECTIONS = _flag('YLM_EVAL_REFLECTIONS', False)
RUN_EXTRAS = _flag('YLM_RUN_EXTRAS', False)
RUN_CPU_BENCH = _flag('YLM_RUN_CPU_BENCH', False)
HEARTBEAT_SECONDS = 120
# -----------------------------------------------------------------------------

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', force_remount=True)
REPO = 'YLM'
SAVE_SUBDIR = 'v0_multiseed' if DATA_SOURCE == 'v0_release' else 'options_bc_test'
SAVE = Path('/content/drive/MyDrive') / f'{REPO}_runs' / SAVE_SUBDIR
SAVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/YLM')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/yaniguan/YLM.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, 'origin/' + BRANCH], cwd=ROOT, check=True)
print(subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout)
os.chdir(ROOT)
pkgs = ['e3nn==0.6.0', 'numpy<2.3', 'scipy', 'rdkit', 'transformers>=4.40,<5', 'accelerate', 'pytest', 'matplotlib']
if not FULL or DATA_SOURCE == 'rebuild':
    pkgs.append('tblite')
subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs], check=True)
import torch
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(), '| mode:', MODE, '| seeds:', SEEDS, '| saving to:', SAVE)
(SAVE / 'runs').mkdir(exist_ok=True)

LOCAL_LOGS = Path('/content/ylm_logs')
LOCAL_LOGS.mkdir(exist_ok=True)

def run_script(script, *args):
    command = [sys.executable, '-u', str(ROOT / 'scripts' / script), *map(str, args)]
    print(' '.join(command), flush=True)
    with (SAVE / 'commands.log').open('a') as log:
        log.write(' '.join(command) + '\n')
    out_path = LOCAL_LOGS / f"{Path(script).stem}_{time.strftime('%H%M%S')}.log"
    with out_path.open('w') as out:
        proc = subprocess.Popen(command, stdout=out, stderr=subprocess.STDOUT)
        while True:
            try:
                code = proc.wait(timeout=HEARTBEAT_SECONDS)
                break
            except subprocess.TimeoutExpired:
                lines = out_path.read_text(errors='replace').splitlines()
                print(time.strftime('%H:%M:%S'), script, '|', lines[-1][:200] if lines else '(no output yet)', flush=True)
    tail = out_path.read_text(errors='replace').splitlines()[-15:]
    print('\n'.join(tail), flush=True)
    if code != 0:
        raise subprocess.CalledProcessError(code, command)


## Official TensorQA-v0 data and frozen text caches
Unpacks and checks the pinned `v0.1.0` `tensorqa_v0.tar.gz` release (`84,757` questions, `584` unique strings) in full mode, caches in Drive, and computes missing frozen text embeddings (`minilm` and/or `matscibert`).


In [ ]:
import urllib.request, zipfile

CACHE = (SAVE / 'v0_data') if (FULL and DATA_SOURCE == 'v0_release') else (SAVE if FULL else SAVE / 'smoke_data')
CACHE.mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)

needed_texts = [TEXT_ENCODER] if MODE == 'options' else [
    t.strip() for t in FRACTION_TEXTS.split(',') if t.strip() and t.strip() != 'taskid'
]
cache_files = ['qm9_xtb.pkl', 'tensorqa_v0.json'] + [f'text_{t}.pt' for t in needed_texts if t != 'taskid']
for name in cache_files:
    if (CACHE / name).exists():
        shutil.copy2(CACHE / name, Path('data') / name)

if FULL and DATA_SOURCE == 'v0_release':
    if not Path('data/qm9_xtb.pkl').exists() or not Path('data/tensorqa_v0.json').exists():
        tar_cache = CACHE / 'tensorqa_v0.tar.gz'
        run_script('06b_unpack_release.py', '--tarball', tar_cache, '--download',
                   '--labels-out', 'data/qm9_xtb.pkl', '--questions-out', 'data/tensorqa_v0.json')
        shutil.copy2('data/qm9_xtb.pkl', CACHE / 'qm9_xtb.pkl')
        shutil.copy2('data/tensorqa_v0.json', CACHE / 'tensorqa_v0.json')
    subprocess.run([sys.executable, '-m', 'src.provenance', '--check-v0', 'data/tensorqa_v0.json'], check=True)
else:
    n_molecules = 20000 if FULL else 500
    if not Path('data/qm9_xtb.pkl').exists():
        archive = Path('data/qm9.zip')
        if not Path('data/gdb9.sdf').exists():
            urllib.request.urlretrieve('https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/molnet_publish/qm9.zip', archive)
            with zipfile.ZipFile(archive) as z:
                z.extractall('data')
        run_script('01_make_labels.py', '--n', n_molecules, '--workers', os.cpu_count() or 2, '--seed', 0)
        shutil.copy2('data/qm9_xtb.pkl', CACHE / 'qm9_xtb.pkl')
    if not Path('data/tensorqa_v0.json').exists():
        run_script('02_make_questions.py', '--seed', 0)
        shutil.copy2('data/tensorqa_v0.json', CACHE / 'tensorqa_v0.json')

for t in needed_texts:
    if t != 'taskid' and not Path(f'data/text_{t}.pt').exists():
        run_script('03_embed_text.py', '--name', t)
        shutil.copy2(f'data/text_{t}.pt', CACHE / f'text_{t}.pt')

subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_provenance.py', '-q'], check=True)


## Train and evaluate across seeds
Each run writes `results.json` with a `provenance` block (`questions_sha256`, `split_counts`, `eval_rotations`, `eval_reflections`, `git_commit`). Plain Option B and Option B + `--attn_sup` use distinct run names so neither overwrites the other.


In [ ]:
P = ('v0_' if DATA_SOURCE == 'v0_release' else 'full_') if FULL else 'tiny_'
eval_flags = ['--eval_batch_size', 32, '--eval_rotations', EVAL_ROTATIONS]
if EVAL_REFLECTIONS:
    eval_flags.append('--eval_reflections')

experiments = []
if MODE == 'options':
    epochs = 60 if FULL else 20
    base = (['--device', 'cuda', '--epochs', epochs] if FULL else
            ['--tiny', '--device', 'cuda', '--epochs', epochs, '--max_steps', 400, '--batch_size', 16]) + eval_flags
    wanted = [o.strip().upper() for o in OPTIONS_TO_RUN.split(',') if o.strip()]
    for seed in SEEDS:
        for o in wanted:
            extra = ['--seed', seed]
            if o == 'C' and C_TP_MODE != 'fully_connected':
                extra += ['--c_tp_mode', C_TP_MODE]
                name = f'{P}optC_{C_TP_MODE}_{TEXT_ENCODER}_s{seed}'
            else:
                name = f'{P}opt{o}_{TEXT_ENCODER}_s{seed}'
            experiments.append((name, TEXT_ENCODER, o, base + extra))
            if o == 'B' and INCLUDE_B_ATTN_SUP and ATTN_SUP > 0:
                tag = format(ATTN_SUP, 'g').replace('.', 'p')
                sup_name = f'{P}optB_attn{tag}_{TEXT_ENCODER}_s{seed}'
                experiments.append((sup_name, TEXT_ENCODER, 'B', base + ['--seed', seed, '--attn_sup', format(ATTN_SUP, 'g')]))
        if RUN_EXTRAS:
            experiments.append((f'{P}taskid_s{seed}', 'taskid', 'A', base + ['--seed', seed]))
            experiments.append((f'{P}optA_legacy_{TEXT_ENCODER}_s{seed}', TEXT_ENCODER, 'A', base + ['--seed', seed, '--legacy_odd_scalars']))
elif MODE == 'fractions':
    epochs = 150 if FULL else 20
    base = (['--device', 'cuda', '--epochs', epochs] if FULL else
            ['--tiny', '--device', 'cuda', '--epochs', epochs, '--max_steps', 400, '--batch_size', 16]) + eval_flags
    fracs = [float(x.strip()) for x in FRACTIONS.split(',') if x.strip()]
    texts = [t.strip() for t in FRACTION_TEXTS.split(',') if t.strip()]
    for seed in SEEDS:
        for f in fracs:
            for t in texts:
                name = f'frac_{t}_f{f:g}_s{seed}' if FULL else f'tiny_frac_{t}_f{f:g}_s{seed}'
                experiments.append((name, t, 'A', base + ['--train_fraction', f, '--seed', seed]))
else:
    raise ValueError(f'Unknown YLM_MODE: {MODE}')

def status():
    for name, *_ in experiments:
        d = SAVE / 'runs' / name
        log = (d / 'log.txt').read_text().splitlines() if (d / 'log.txt').exists() else []
        last = [l for l in log if l.startswith('epoch ')][-1:] or ['(no epochs yet)']
        print(f"{name}: finished={(d / 'results.json').exists()} | last: {last[0]}")

status()
for name, text, option, flags in experiments:
    run_dir = SAVE / 'runs' / name
    if (run_dir / 'results.json').exists():
        print('Completed; skipping', name)
        continue
    run_script('04_train.py', '--resume', '--output_dir', run_dir, '--name', name, '--text', text, '--option', option, *flags)

json.dump(dict(mode=MODE, full=FULL, data_source=DATA_SOURCE, seeds=SEEDS,
               eval_rotations=EVAL_ROTATIONS, eval_reflections=EVAL_REFLECTIONS,
               branch=BRANCH, attn_sup=ATTN_SUP, include_b_attn_sup=INCLUDE_B_ATTN_SUP,
               gpu=torch.cuda.get_device_name()),
          (SAVE / 'config.json').open('w'), indent=2)
status()


## Build provenance-checked Markdown tables and plots
Writes `results_options_v0.md` + `results_options_v0.json` in `options` mode, or combined multi-seed `results.md` + `sample_efficiency.png` in `fractions` mode.


In [ ]:
if MODE == 'options':
    extra = [] if (FULL and DATA_SOURCE == 'v0_release' and not EVAL_REFLECTIONS and EVAL_ROTATIONS == 32 and len(SEEDS) >= 3) else [
        '--allow-unverified', '--min-seeds', '1', '--eval-rotations', str(EVAL_ROTATIONS)
    ]
    out_md = SAVE / ('results_options_v0.md' if FULL else 'smoke_results_options_v0.md')
    out_json = SAVE / ('results_options_v0.json' if FULL else 'smoke_results_options_v0.json')
    run_script('17_make_options_table.py', SAVE / 'runs', '--out', out_md, '--json', out_json, *extra)
    print(out_md.read_text())
elif MODE == 'fractions':
    folders = (['runs_slurm', str(SAVE / 'runs')] if FULL else [str(SAVE / 'runs')])
    out_md = SAVE / ('results.md' if FULL else 'smoke_results.md')
    out_analysis = SAVE / ('analysis' if FULL else 'smoke_analysis')
    run_script('07_make_table.py', *folders, '--out', out_md)
    run_script('08_analyze.py', '--runs', *folders, '--out', out_analysis, '--sample_efficiency_only')
    print(out_md.read_text())

if RUN_CPU_BENCH:
    run_script('bench_full_model.py', '--threads', 2, '--output', SAVE / 'cpu_full_model_benchmark.json')
